### Imports e Configuração de Ambiente

In [25]:
import os
import duckdb
import numpy as np
import pandas as pd
from optbinning import OptimalBinning

# Ajusta o diretório de execução para a raiz do projeto
if os.getcwd().endswith("notebooks"):
  os.chdir("..")

con = duckdb.connect()
print("✅ Ambiente e DuckDB inicializados!")

✅ Ambiente e DuckDB inicializados!


### Consolidação Visão Contrato e Visão Cliente (SQL)

In [26]:
# 1. Agregações das tabelas secundárias (Visão Contrato)
con.execute("""
    CREATE OR REPLACE VIEW vw_bureau_agg AS
    SELECT 
        SK_ID_CURR AS id_cliente,
        COUNT(SK_ID_BUREAU) AS bureau_qtd_creditos,
        COUNT(CASE WHEN CREDIT_ACTIVE = 'Active' THEN 1 END) AS bureau_qtd_ativos,
        ROUND(SUM(COALESCE(AMT_CREDIT_SUM, 0)), 2) AS bureau_total_credito,
        ROUND(SUM(COALESCE(AMT_CREDIT_SUM_DEBT, 0)), 2) AS bureau_total_divida,
        ROUND(AVG(COALESCE(DAYS_CREDIT, 0)), 2) AS bureau_media_dias_credito
    FROM 'data/bureau.parquet'
    GROUP BY SK_ID_CURR;
""")

con.execute("""
    CREATE OR REPLACE VIEW vw_prev_app_agg AS
    SELECT 
        SK_ID_CURR AS id_cliente,
        COUNT(SK_ID_PREV) AS prev_qtd_pedidos,
        COUNT(CASE WHEN NAME_CONTRACT_STATUS = 'Approved' THEN 1 END) AS prev_qtd_aprovados,
        COUNT(CASE WHEN NAME_CONTRACT_STATUS = 'Refused' THEN 1 END) AS prev_qtd_recusados,
        ROUND(AVG(COALESCE(AMT_APPLICATION, 0)), 2) AS prev_media_valor_solicitado
    FROM 'data/previous_application.parquet'
    GROUP BY SK_ID_CURR;
""")

# 2. Consolidação da ABT de Treino + Limpeza de Anomalias
df_abt_treino = con.execute("""
    SELECT 
        app.SK_ID_CURR AS id_cliente,
        app.TARGET AS target,
        app.NAME_CONTRACT_TYPE,
        app.CODE_GENDER,
        app.FLAG_OWN_CAR,
        app.FLAG_OWN_REALTY,
        app.AMT_INCOME_TOTAL,
        app.AMT_CREDIT,
        app.AMT_ANNUITY,
        app.AMT_GOODS_PRICE,
        app.DAYS_BIRTH,
        app.EXT_SOURCE_1,
        app.EXT_SOURCE_2,
        app.EXT_SOURCE_3,
        
        -- Tratamento de Anomalia
        CASE WHEN app.DAYS_EMPLOYED = 365243 THEN NULL ELSE app.DAYS_EMPLOYED END AS DAYS_EMPLOYED,
        
        -- Features Agregadas
        COALESCE(b.bureau_qtd_creditos, 0) AS bureau_qtd_creditos,
        COALESCE(b.bureau_qtd_ativos, 0) AS bureau_qtd_ativos,
        COALESCE(b.bureau_total_credito, 0) AS bureau_total_credito,
        COALESCE(b.bureau_total_divida, 0) AS bureau_total_divida,
        COALESCE(b.bureau_media_dias_credito, 0) AS bureau_media_dias_credito,
        COALESCE(p.prev_qtd_pedidos, 0) AS prev_qtd_pedidos,
        COALESCE(p.prev_qtd_aprovados, 0) AS prev_qtd_aprovados,
        COALESCE(p.prev_qtd_recusados, 0) AS prev_qtd_recusados,
        COALESCE(p.prev_media_valor_solicitado, 0) AS prev_media_valor_solicitado
        
    FROM 'data/application_train.parquet' app
    LEFT JOIN vw_bureau_agg b ON app.SK_ID_CURR = b.id_cliente
    LEFT JOIN vw_prev_app_agg p ON app.SK_ID_CURR = p.id_cliente
""").df()

print(f"✅ Base Consolidada de Treino: {df_abt_treino.shape}")

✅ Base Consolidada de Treino: (307511, 24)


### OptBinning, Cálculo de IV e Transformação WoE

In [28]:
colunas_ignorar = ["id_cliente", "target", "sk_id_curr"]
colunas_analise = [
    c for c in df_abt_treino.columns if c.lower() not in colunas_ignorar
]

mapa_iv = {}
pdf_treino_woe = pd.DataFrame()

print("⏳ Processando binnagem otimizada e WoE...")

for col in colunas_analise:
  dtype = "categorical" if df_abt_treino[col].dtype == "object" else "numerical"
  optb = OptimalBinning(name=col, dtype=dtype, solver="cp")

  try:
    optb.fit(df_abt_treino[col], df_abt_treino["target"])
    iv = optb.binning_table.build()["IV"].to_list()[-1]
    mapa_iv[col] = iv

    # Aplica transformação WoE
    pdf_treino_woe[f"{col}_woe"] = optb.transform(
        df_abt_treino[col], metric="woe"
    )
  except Exception as e:
    mapa_iv[col] = 0.0

# Filtro de Variáveis Relevantes (0.02 <= IV < 0.50)
features_relevantes = [
    f"{col}_woe" for col, iv in mapa_iv.items() if 0.02 <= iv < 0.50
]

print(
    f"✅ Binning concluído! Features selecionadas por IV:"
    f" {len(features_relevantes)} de {len(colunas_analise)}"
)

⏳ Processando binnagem otimizada e WoE...


✅ Binning concluído! Features selecionadas por IV: 12 de 22


### Relatório de Information Value (IV)

In [29]:
# Tabela de classificação das variáveis por IV
df_iv_report = (
    pd.DataFrame(list(mapa_iv.items()), columns=["Variavel", "IV"])
    .sort_values(by="IV", ascending=False)
    .reset_index(drop=True)
)

print("--- Top 10 Features por Poder Preditivo (IV) ---")
display(df_iv_report.head(10))

--- Top 10 Features por Poder Preditivo (IV) ---


,Variavel,IV
0,EXT_SOURCE_3,0.335262
1,EXT_SOURCE_2,0.321683
2,EXT_SOURCE_1,0.146393
3,bureau_media_dias_credito,0.126020
4,DAYS_EMPLOYED,0.114054
5,AMT_GOODS_PRICE,0.092037
6,DAYS_BIRTH,0.087245
7,AMT_CREDIT,0.059368
8,prev_qtd_recusados,0.053781
9,AMT_ANNUITY,0.031179


### Recomposição e Exportação do Dataset Refinado

In [30]:
# Preserva ID do cliente, Target e apenas as features filtradas por IV
df_final_woe = pd.concat(
    [
        df_abt_treino[["id_cliente", "target"]].reset_index(drop=True),
        pdf_treino_woe[features_relevantes].reset_index(drop=True),
    ],
    axis=1,
)

# Salva a base processada em Parquet para o Notebook 03
df_final_woe.to_parquet("data/train_pipeline_woe.parquet", index=False)

print(f"🎉 Pipeline concluída!")
print(f"📁 Arquivo 'data/train_pipeline_woe.parquet' gerado com sucesso.")
print(f"📐 Dimensões finais: {df_final_woe.shape}")

🎉 Pipeline concluída!
📁 Arquivo 'data/train_pipeline_woe.parquet' gerado com sucesso.
📐 Dimensões finais: (307511, 14)
